In [1]:
import pandas as pd

from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder
from sklearn.preprocessing import StandardScaler

In [4]:
TRAIN_PATH = "/Users/mitulshah/Downloads/mlops-churn-project/data/processed/train.csv"
TEST_PATH = "/Users/mitulshah/Downloads/mlops-churn-project/data/processed/test.csv"

In [5]:
train_df = pd.read_csv(TRAIN_PATH)
test_df = pd.read_csv(TEST_PATH)

In [6]:
X_train = train_df.drop(columns=["churn"])
y_train = train_df["churn"]

X_test = test_df.drop(columns=["churn"])
y_test = test_df["churn"]

In [7]:
NUMERICAL_FEATURES = [
    "age",
    "tenure_months",
    "monthly_charges",
    "customer_service_calls"
]

In [8]:
CATEGORICAL_FEATURES = [
    "contract_type",
    "internet_service",
    "tech_support",
    "online_security",
    "payment_method"
]

In [9]:
numerical_pipeline = Pipeline(
    steps=[
        ("imputer", SimpleImputer(strategy="median")),
        ("scaler", StandardScaler())
    ]
)

In [10]:
categorical_pipeline = Pipeline(
    steps=[
        (
            "imputer",
            SimpleImputer(strategy="most_frequent")
        ),
        (
            "onehot",
            OneHotEncoder(
                handle_unknown="ignore",
                sparse_output=False
            )
        )
    ]
)

In [11]:
preprocessor = ColumnTransformer(
    transformers=[
        (
            "numerical",
            numerical_pipeline,
            NUMERICAL_FEATURES
        ),
        (
            "categorical",
            categorical_pipeline,
            CATEGORICAL_FEATURES
        )
    ]
)

In [12]:
X_train_processed = preprocessor.fit_transform(X_train)

In [13]:
X_test_processed = preprocessor.transform(X_test)

In [14]:
print("Original training shape:", X_train.shape)
print("Processed training shape:", X_train_processed.shape)

print("Original test shape:", X_test.shape)
print("Processed test shape:", X_test_processed.shape)

Original training shape: (4000, 10)
Processed training shape: (4000, 17)
Original test shape: (1000, 10)
Processed test shape: (1000, 17)


In [15]:
feature_names = preprocessor.get_feature_names_out()

print(feature_names)

['numerical__age' 'numerical__tenure_months' 'numerical__monthly_charges'
 'numerical__customer_service_calls'
 'categorical__contract_type_Month-to-month'
 'categorical__contract_type_One year'
 'categorical__contract_type_Two year' 'categorical__internet_service_DSL'
 'categorical__internet_service_Fiber optic'
 'categorical__internet_service_No' 'categorical__tech_support_No'
 'categorical__tech_support_Yes' 'categorical__online_security_No'
 'categorical__online_security_Yes'
 'categorical__payment_method_Bank transfer'
 'categorical__payment_method_Credit card'
 'categorical__payment_method_Electronic check']


In [16]:
X_train_processed_df = pd.DataFrame(
    X_train_processed,
    columns=feature_names,
    index=X_train.index
)

X_test_processed_df = pd.DataFrame(
    X_test_processed,
    columns=feature_names,
    index=X_test.index
)

In [17]:
import os

os.makedirs("/Users/mitulshah/Downloads/mlops-churn-project/data/processed/features", exist_ok=True)

In [18]:
X_train_processed_df.to_csv(
    "/Users/mitulshah/Downloads/mlops-churn-project/data/processed/features/X_train.csv",
    index=False
)

X_test_processed_df.to_csv(
    "/Users/mitulshah/Downloads/mlops-churn-project/data/processed/features/X_test.csv",
    index=False
)

y_train.to_csv(
    "/Users/mitulshah/Downloads/mlops-churn-project/data/processed/features/y_train.csv",
    index=False
)

y_test.to_csv(
    "/Users/mitulshah/Downloads/mlops-churn-project/data/processed/features/y_test.csv",
    index=False
)

In [19]:
import joblib

In [20]:
joblib.dump(
    preprocessor,
    "/Users/mitulshah/Downloads/mlops-churn-project/models/preprocessor.pkl"
)

['/Users/mitulshah/Downloads/mlops-churn-project/models/preprocessor.pkl']

In [21]:
print("Training data:")
print(X_train_processed_df.shape)

print("\nTesting data:")
print(X_test_processed_df.shape)

print("\nFeature names:")
print(feature_names)

print("\nPreprocessor saved successfully.")

Training data:
(4000, 17)

Testing data:
(1000, 17)

Feature names:
['numerical__age' 'numerical__tenure_months' 'numerical__monthly_charges'
 'numerical__customer_service_calls'
 'categorical__contract_type_Month-to-month'
 'categorical__contract_type_One year'
 'categorical__contract_type_Two year' 'categorical__internet_service_DSL'
 'categorical__internet_service_Fiber optic'
 'categorical__internet_service_No' 'categorical__tech_support_No'
 'categorical__tech_support_Yes' 'categorical__online_security_No'
 'categorical__online_security_Yes'
 'categorical__payment_method_Bank transfer'
 'categorical__payment_method_Credit card'
 'categorical__payment_method_Electronic check']

Preprocessor saved successfully.
